# Performance metrics

In [94]:
import pandas as pd 
import matplotlib.pyplot as pyplot
import seaborn as sns
import os

In [95]:
df = pd.read_csv("clean_data/exp_web_merged.csv")
df_all= pd.read_csv('clean_data/all_data_merged.csv')

In [96]:
df.columns

Index(['client_id', 'variation', 'visitor_id', 'visit_id', 'process_step',
       'date_time'],
      dtype='object')

In [97]:
df_all.columns

Index(['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time',
       'variation', 'client_tenure_yr', 'client_tenure_month', 'client_age',
       'gender', 'num_accts', 'balance', 'calls_6_mnth', 'logons_6_mnth'],
      dtype='object')

# KPI 1 : Completion Rate

**Defintion** : measures the percentage of visits that reach the Confirm step after progressing through all funnel steps from Start, including visits where users navigate back one or more steps before completing the journey.

In [98]:
df_sorted = df.sort_values(['client_id', 'date_time'])

In [99]:
ordered_steps = df_sorted.groupby('client_id')['process_step'].apply(list)

In [100]:
required_order = ['start', 'step_1','step_2','step_3','confirm']

#Returns True if the required steps appear in the correct order within the user's sequence. Repeats and loops are allowed between steps.

def is_complete_in_order(steps_sequence, required=required_order):
    required_index = 0
    
    for step in steps_sequence:
        if step == required[required_index]:
            required_index += 1
            if required_index == len(required):
                return True
    
    return False

In [101]:
completed = ordered_steps.apply(is_complete_in_order).astype(int)

In [102]:
completion_rate = completed.mean()
completion_rate

np.float64(0.6638613861386139)

In [103]:
completion_rate_by_variation = completed.groupby(df_sorted.groupby('client_id')['variation'].first()).mean()
completion_rate_by_variation

variation
Control    0.645674
Test       0.679732
Name: process_step, dtype: float64

In [104]:
completion_rate_v2 = (completion_rate_by_variation * 100).round(2)
completion_rate_v2

variation
Control    64.57
Test       67.97
Name: process_step, dtype: float64

#### Adding a new column completed funnel
(will be used for Hypothesis testing)

In [105]:
df_completed = completed.reset_index()
df_completed.columns = ['client_id','completed']

In [106]:
visit_variation = df[['client_id', 'variation']].drop_duplicates()

In [107]:
visit_completed = visit_variation.merge(df_completed, on='client_id', how='left')
visit_completed

,client_id,variation,completed
0,9988021,Test,0
1,8320017,Test,1
2,4033851,Control,1
3,1982004,Test,1
4,9294070,Control,0
...,...,...,...
50495,393005,Control,1
50496,2908510,Control,0
50497,7230446,Test,0
50498,5230357,Test,1


In [108]:
#visit_completed.to_csv("clean_data/visit_completed.csv",index=False)

# KPI 2 : Time spent per step

In [109]:
df = df.sort_values(['visit_id', 'date_time'])
df.head()

,client_id,variation,visitor_id,visit_id,process_step,date_time
140614,3561384,Test,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17
140613,3561384,Test,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09
311202,7338123,Test,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56
311201,7338123,Test,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12
311200,7338123,Test,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21


In [110]:
df['next_time'] = df.groupby('visit_id')['date_time'].shift(-1)

In [111]:
df['date_time'] = pd.to_datetime(df['date_time'])
df['next_time'] = pd.to_datetime(df['next_time'])

In [112]:
df['time_spent_clean'] = (df['next_time'] - df['date_time']).dt.total_seconds()

In [113]:
# handeling last step ( confirm) 
df = df[df['time_spent_clean'].notna()]

In [114]:
# fixing repeated steps
step_time_per_visit = (
    df.groupby(['visit_id', 'process_step'])['time_spent_clean']
    .sum()
    .reset_index())

### Handeling time spent outliers

In [115]:
print(df['time_spent_clean'].describe())

count    248030.000000
mean         84.207939
std         215.986776
min           0.000000
25%          13.000000
50%          36.000000
75%          83.000000
max       40235.000000
Name: time_spent_clean, dtype: float64


In [116]:
print((df['time_spent_clean']/60).describe().round(2))

count    248030.00
mean          1.40
std           3.60
min           0.00
25%           0.22
50%           0.60
75%           1.38
max         670.58
Name: time_spent_clean, dtype: float64


In [117]:
(df['time_spent_clean'] / 60 > 15).value_counts()

time_spent_clean
False    245224
True       2806
Name: count, dtype: int64

In [118]:
# converting time spent to minutes
df['time_spent_mn'] = df['time_spent_clean']/60

In [119]:
# Count of outliers per step ( above 15min per step)
outliers = df[df['time_spent_mn'] > 15].groupby('process_step')['time_spent_mn'].agg(['count', 'mean', 'max','min']).round(2)
print(outliers)

              count   mean     max    min
process_step                             
confirm         562  21.05  243.02  15.02
start           854  24.82  670.58  15.02
step_1          310  24.54  171.43  15.12
step_2          248  21.13  362.72  15.03
step_3          832  20.74  111.53  15.02


In [120]:
# Drop anything above 15 minutes per step
df_15 = df[df['time_spent_mn'] <= 15]

In [121]:
final_step_time = (df_15.groupby('process_step')['time_spent_clean']
                   .mean()
                   .reset_index())
final_step_time

,process_step,time_spent_clean
0,confirm,125.043087
1,start,48.793444
2,step_1,49.268610
3,step_2,84.860655
4,step_3,111.808901


In [122]:
# final aggregation
final_step_time = (df_15.groupby(['variation','process_step',])['time_spent_mn']
                   .mean()
                   .reset_index()
                   .round(2)
                   .pivot(index='process_step', columns='variation', values='time_spent_mn'))
final_step_time

variation,Control,Test
process_step,,
confirm,1.68,2.29
start,0.87,0.77
step_1,0.72,0.90
step_2,1.46,1.38
step_3,1.99,1.76


In [123]:
final_step_time = (df_15.groupby('variation')['time_spent_mn']
                   .mean()
                   .reset_index()
                   .round(2))
final_step_time

,variation,time_spent_mn
0,Control,1.19
1,Test,1.14


# KPI 3 : Error rate

In [124]:
df_sorted = df.sort_values(['visit_id', 'date_time']).reset_index(drop=True)

In [125]:
# 2. Map each step to a numeric rank
step_rank = {'start': 0, 'step_1': 1, 'step_2': 2, 'step_3': 3, 'confirm': 4}
df_sorted['step_rank'] = df_sorted['process_step'].map(step_rank)

In [126]:
df_sorted.head(5)

,client_id,variation,visitor_id,visit_id,process_step,date_time,next_time,time_spent_clean,time_spent_mn,step_rank
0,3561384,Test,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,2017-04-26 13:23:09,52.0,0.866667,4
1,7338123,Test,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,2017-04-09 16:21:12,16.0,0.266667,0
2,7338123,Test,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,2017-04-09 16:21:21,9.0,0.150000,1
3,7338123,Test,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,2017-04-09 16:21:35,14.0,0.233333,2
4,7338123,Test,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:35,2017-04-09 16:21:41,6.0,0.100000,1


In [127]:
# 3. Get the previous step and its rank within each visit
df_sorted['prev_step'] = df_sorted.groupby('visit_id')['process_step'].shift(1)
df_sorted['prev_rank'] = df_sorted.groupby('visit_id')['step_rank'].shift(1)


In a funnel, going **backward** means you were at a higher step and dropped to a lower one :

- `prev_rank = 3`, `step_rank = 1` → user dropped back → `3 - 1 = 2` ✅
- `prev_rank = 1`, `step_rank = 3` → user moved **forward** → `1 - 3 = -2` → clamped to 0 ✅

So `prev_rank - step_rank` makes perfect sense here :
- **Positive result** = user went backward in the funnel
- **Negative result** = user moved forward (not a backward distance, so set to 0)
It does **not** remove the row — the row stays in your data.

Setting it to 0 just means : **"this user didn't go backward, so their backward distance is 0"**

Think of it like this :

| user | prev_rank | step_rank | backward_distance |
|------|-----------|-----------|-------------------|
| A | 3 | 1 | 2 (went backward) |
| B | 1 | 3 | **0** (moved forward, no backward distance) |
| C | 2 | 2 | **0** (stayed at same step) |

User B and C are still in your dataset, they just contribute **0** to any backward distance calculation — which makes sense, they didn't drop back.

So when you later do a sum or average of `backward_distance`, forward-moving users won't inflate the result, but they're still counted in your data.

In [128]:
# 4. Compute backward distance (weighted: step_3 -> step_1 = 2 errors)
df_sorted['backward_distance'] = df_sorted['prev_rank'] - df_sorted['step_rank']

In [129]:
df_sorted.loc[df_sorted['backward_distance'] < 0, 'backward_distance'] = 0
df_sorted['backward_distance'] = df_sorted['backward_distance'].fillna(0)

In [130]:
# A.Overall weighted error rate
overall_error_rate = df_sorted['backward_distance'].sum()*100 / len(df_sorted)
overall_error_rate

np.float64(11.563520541869934)

In [131]:
# B. Overall error rate per variation
error_by_variation = df_sorted.groupby('variation')['backward_distance'].mean()
error_by_variation = (error_by_variation * 100 ).round(2)
error_by_variation


variation
Control    11.13
Test       11.90
Name: backward_distance, dtype: float64

In [132]:
error_by_step = (
    df_sorted.groupby('prev_step')['backward_distance']
    .agg(total_errors='sum', error_rate='mean')
    .reindex(['start', 'step_1', 'step_2', 'step_3', 'confirm'])
)
error_by_step

,total_errors,error_rate
prev_step,,
start,0.0,0.000000
step_1,6378.0,0.113385
step_2,6942.0,0.141992
step_3,12394.0,0.995742
confirm,2967.0,1.277778


In [133]:
error_by_step_and_variation = (
    df_sorted.groupby(['variation', 'prev_step'])['backward_distance']
    .mean().round(2)
    .unstack('variation')
    .reindex(['start', 'step_1', 'step_2', 'step_3', 'confirm'])
)
error_by_step_and_variation

variation,Control,Test
prev_step,,
start,0.00,0.00
step_1,0.07,0.14
step_2,0.11,0.17
step_3,1.06,0.94
confirm,1.94,0.79


### Conclusion ###

The new design (test variation) performs better on Step 3 and the Confirm step, with a lower average backward navigation rate compared to the control variation. However, Steps 1 and 2 show a slightly higher average error rate in the test variation.


### Error rate by demographic

In [134]:
df_sorted.columns

Index(['client_id', 'variation', 'visitor_id', 'visit_id', 'process_step',
       'date_time', 'next_time', 'time_spent_clean', 'time_spent_mn',
       'step_rank', 'prev_step', 'prev_rank', 'backward_distance'],
      dtype='object')

# KPI 4 : Return rate

**Definition**

the percentage of clients who needed more than one visit to complete the funnel

In [135]:
# Group by client and variation, count unique visit_ids per client
#  how many sessions each client had
sessions_per_client = (df.groupby(['client_id', 'variation'])['visit_id']
                       .nunique()
                       .reset_index())
sessions_per_client

,client_id,variation,visit_id
0,555,Test,1
1,647,Test,1
2,934,Test,1
3,1028,Control,1
4,1186,Control,1
...,...,...,...
46559,9999150,Test,1
46560,9999400,Test,1
46561,9999626,Test,1
46562,9999729,Test,3


In [136]:
# Flag clients who came back more than once
# True = returned, False = only visited once
sessions_per_client['is_return'] = sessions_per_client['visit_id'] > 1

In [137]:
(sessions_per_client.groupby('variation')['is_return'].mean() * 100).round(2)

variation
Control    18.11
Test       18.37
Name: is_return, dtype: float64

In [138]:
sessions_per_client.groupby('variation')['is_return'].sum()

variation
Control    3824
Test       4675
Name: is_return, dtype: int64

**Conclusion** : client  who saw the test variation needed more than 1 visit to complete the funnel which is (0,2%) more than the control variation

# Analysis by demographics segments

**1- Creating the demographic groups**

In [139]:
df_all.columns

Index(['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time',
       'variation', 'client_tenure_yr', 'client_tenure_month', 'client_age',
       'gender', 'num_accts', 'balance', 'calls_6_mnth', 'logons_6_mnth'],
      dtype='object')

In [140]:
# Age group
df_all['age_group'] = pd.cut(
    df_all['client_age'],
    bins=[18, 34, 48, 60, 90],
    labels=['Young', 'Middle', 'Senior', 'Elders']
)

In [141]:
# Tenure group
df_all['tenure_group'] = pd.cut(
    df_all['client_tenure_yr'],
    bins=[0, 5, 10, 100],
    labels=['New', 'Mid', 'Long']
)

In [142]:
df_all['tenure_group'].value_counts()

tenure_group
Long    163602
Mid      99371
New      54150
Name: count, dtype: int64

In [143]:
# Logon group
def classify_logon(x):
    if x == 0:
        return '0'
    else:
        return '1'

df_all['logon_group'] = df_all['logons_6_mnth'].apply(classify_logon)

In [144]:
# Balance group
def classify_balance(x):
    if x < 50000:
        return 'Low (<50K)'
    elif x < 100000:
        return 'Mid (50-100K)'
    elif x < 250000:
        return 'High (100-250K)'
    else:
        return 'Very High (250K+)'

df_all['balance_group'] = df_all['balance'].apply(classify_balance)

In [145]:
df_all[['gender','client_age', 'age_group', 'client_tenure_yr', 'tenure_group',
              'logons_6_mnth', 'logon_group', 'balance', 'balance_group']].head(10)

,gender,client_age,age_group,client_tenure_yr,tenure_group,logons_6_mnth,logon_group,balance,balance_group
0,Unknown,79,Elders,5.0,New,4,1,189023.86,High (100-250K)
1,Unknown,79,Elders,5.0,New,4,1,189023.86,High (100-250K)
2,Unknown,79,Elders,5.0,New,4,1,189023.86,High (100-250K)
3,Unknown,79,Elders,5.0,New,4,1,189023.86,High (100-250K)
4,Unknown,79,Elders,5.0,New,4,1,189023.86,High (100-250K)
5,Unknown,79,Elders,5.0,New,4,1,189023.86,High (100-250K)
6,Unknown,79,Elders,5.0,New,4,1,189023.86,High (100-250K)
7,Unknown,79,Elders,5.0,New,4,1,189023.86,High (100-250K)
8,Male,34,Young,22.0,Long,8,1,36001.90,Low (<50K)
9,Male,34,Young,22.0,Long,8,1,36001.90,Low (<50K)


**2- extracting the demographics per client data**

In [146]:
df_demo_cols = (df_all[['client_id', 'gender', 'age_group',
                         'tenure_group', 'logon_group', 'balance_group']]
                .drop_duplicates(subset='client_id'))
df_demo_cols.shape

(50487, 6)

**3- Merging demographics data into the KPI dataframe**

In [147]:
# Error rate
df_sorted = df_sorted.merge(df_demo_cols, on='client_id', how='left')
# Time per step
df_15 = df_15.merge(df_demo_cols, on='client_id', how='left')
# Return rate
sessions_per_client = sessions_per_client.merge(df_demo_cols, on='client_id', how='left')


In [148]:
segments = ['gender', 'age_group', 'tenure_group', 'logon_group', 'balance_group']

In [149]:
for seg in segments:
    print(f"\n{'='*50}")
    print(f"SEGMENT: {seg.upper()}")
    print(f"{'='*50}")

    # Error rate — using backward_distance instead of is_backward
    error = (df_sorted.groupby(['variation', seg], observed=True)['backward_distance']
             .mean()).round(2).unstack('variation')
    print(f"\n--- Error Rate (backward_distance) ---")
    print(error)

    # Time per step
    time_step = (df_15.groupby(['variation', seg], observed=True)['time_spent_mn']
                 .mean()).round(2).unstack('variation')
    print(f"\n--- Avg Time per Step (min) ---")
    print(time_step)

    # Return rate
    return_rate = (sessions_per_client.groupby(['variation', seg], observed=True)['is_return']
                   .mean() * 100).round(2).unstack('variation')
    print(f"\n--- Return Rate (%) ---")
    print(return_rate)


SEGMENT: GENDER

--- Error Rate (backward_distance) ---
variation  Control  Test
gender                  
Female        0.11  0.13
Male          0.11  0.12
Unknown       0.12  0.11
X              NaN  0.17

--- Avg Time per Step (min) ---
variation  Control  Test
gender                  
Female        1.21  1.15
Male          1.25  1.17
Unknown       1.12  1.10
X              NaN  0.98

--- Return Rate (%) ---
variation  Control   Test
gender                   
Female       18.91  19.57
Male         18.75  18.90
Unknown      16.72  16.69
X              NaN   0.00

SEGMENT: AGE_GROUP

--- Error Rate (backward_distance) ---
variation  Control  Test
age_group               
Young         0.11  0.09
Middle        0.11  0.10
Senior        0.11  0.13
Elders        0.12  0.15

--- Avg Time per Step (min) ---
variation  Control  Test
age_group               
Young         0.96  0.93
Middle        1.12  1.05
Senior        1.31  1.25
Elders        1.37  1.30

--- Return Rate (%) ---
variation  

**Findings**
1. Senior + Elders → Return Rate gap is the biggest signal
Senior: +1.91 and Elders: +1.43 → older clients need significantly more sessions with the new UI 

2. Young + Middle → New UI clearly works better
Young: -0.79 and Middle: -0.93 on return rate => younger clients need fewer sessions 

3. Time per step → Consistently better in Test across ALL segments
Every single group is faster with the new UI — this is the most consistent and reliable finding 

